# 01. Jupyter 기본기 — 노트북을 '제대로' 쓰는 법

노트북은 편한 만큼 **틀린 결과를 그럴듯하게 보여 주기도 쉽다**. 이 노트북은 문법이 아니라
그 함정들을 다룬다.

## 학습 목표
1. 셀 실행 순서가 결과를 바꾼다 — 숨은 상태(hidden state) 문제
2. 커널과 가상환경이 어긋나는 문제 (`pip install` 했는데 `ModuleNotFoundError`)
3. 매직 커맨드 — `%time`, `%%timeit`, `%pwd`, `!셸명령`
4. 도움말 — `?`, `??`, Tab
5. 노트북 위생 — `.ipynb_checkpoints`, 출력 커밋, git diff 지옥
6. `Untitled.ipynb` 가 왜 문제인가

> 이 저장소의 이전 상태가 위 문제의 표본이었다. `legacy/` 에 그대로 남겨 두었다.

## 1. 지금 이 노트북은 어떤 파이썬으로 도는가

노트북에서 가장 흔한 사고는 **터미널에서 설치한 파이썬과 커널의 파이썬이 다른 것**이다.
`pip install pandas` 를 분명히 했는데 `ModuleNotFoundError: No module named 'pandas'` 가 뜬다면
십중팔구 이 문제다. 먼저 확인부터 한다.

In [1]:
import sys
from pathlib import Path

print("실행 파일 :", sys.executable)
print("버전      :", sys.version.split()[0])
print("작업 폴더 :", Path.cwd())

실행 파일 : /Users/ogu/dev/jupyterTest/.venv/bin/python
버전      : 3.12.13
작업 폴더 : /Users/ogu/dev/jupyterTest/notebooks


`sys.executable` 이 내가 설치한 가상환경(`.venv/bin/python`)을 가리키는지 확인한다.
다르다면 커널을 잘못 고른 것이다. 해결은 둘 중 하나다.

```bash
# ① 가상환경을 커널로 등록한 뒤, 노트북 우상단에서 그 커널을 선택
.venv/bin/python -m pip install ipykernel
.venv/bin/python -m ipykernel install --user --name jupytertest --display-name "Python (jupyterTest)"

# ② 혹은 노트북 안에서 '지금 이 커널의 pip' 으로 설치 (sys.executable 을 쓰는 게 핵심)
```
```python
%pip install pandas      # 권장 — 현재 커널에 설치된다
!pip install pandas      # 위험 — PATH 상의 다른 pip 일 수 있다
```

## 2. 숨은 상태 — 노트북 최대의 함정

셀은 **위에서 아래로** 실행된 것처럼 보이지만, 실제로는 **사용자가 누른 순서대로** 실행된다.
커널 메모리에는 지운 셀의 변수도 그대로 남아 있다.

In [2]:
total = 10
print("total =", total)

total = 10


In [3]:
total = total + 5   # 이 셀을 두 번 실행하면? 20 이 된다
print("total =", total)

total = 15


위 셀을 두 번 누르면 `20`, 세 번 누르면 `25` 가 된다. 코드는 그대로인데 결과가 달라진다.
저장된 노트북의 출력이 `In [7]`, `In [3]`, `In [12]` 처럼 뒤죽박죽이면 **그 출력은 믿을 수 없다.**

### 규율 세 가지
1. 커밋·공유 전에는 반드시 **Kernel → Restart Kernel and Run All Cells**
2. 셀 번호가 위에서부터 1, 2, 3... 순서인지 확인
3. 이미 정의한 변수에 의존하는 셀은 가급적 함수로 감싼다 (아래처럼)

In [4]:
def accumulate(start: int, step: int, times: int) -> int:
    """몇 번을 실행해도 같은 값을 주는 형태 — 노트북에서도 함수가 안전하다."""
    return start + step * times

print(accumulate(10, 5, 1), accumulate(10, 5, 2))

15 20


실행 순서를 눈으로 확인하는 방법도 있다. `In [n]` 의 n 이 실행 카운터다.

In [5]:
ipython = get_ipython()  # 노트북(IPython) 안에서만 존재하는 함수
print("지금까지 실행된 셀 수:", ipython.execution_count)
print("이 값이 셀 개수보다 크면, 어떤 셀을 여러 번 눌렀다는 뜻이다.")

지금까지 실행된 셀 수: 6
이 값이 셀 개수보다 크면, 어떤 셀을 여러 번 눌렀다는 뜻이다.


## 3. 매직 커맨드

`%` 로 시작하면 **줄 매직**, `%%` 로 시작하면 **셀 전체 매직**이다.

In [6]:
%pwd

'/Users/ogu/dev/jupyterTest/notebooks'

In [7]:
%%time
# 셀 전체 실행 시간 (한 번만 측정)
total = sum(i * i for i in range(1_000_000))
print(total)

333332833333500000
CPU times: user 139 ms, sys: 4.23 ms, total: 143 ms
Wall time: 169 ms


In [8]:
%%timeit -n 5 -r 3
# 여러 번 반복해 평균·표준편차까지 — 성능 비교는 %time 이 아니라 이쪽으로
sum(i * i for i in range(200_000))

50.3 ms ± 9.76 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


자주 쓰는 것들:

| 매직 | 용도 |
|---|---|
| `%pwd`, `%cd`, `%ls` | 경로 다루기 |
| `%time`, `%timeit` | 실행 시간 |
| `%who`, `%whos` | 현재 정의된 변수 목록 (숨은 상태 점검!) |
| `%reset -f` | 모든 변수 삭제 (커널 재시작 없이 초기화) |
| `%matplotlib inline` | 그래프를 노트북 안에 그림 (요즘은 기본값) |
| `%load_ext autoreload` + `%autoreload 2` | 외부 .py 수정이 자동 반영 |
| `%pip install`, `%conda install` | 현재 커널에 설치 |
| `!명령` | 셸 명령 실행 |

In [9]:
%whos

Variable     Type                   Data/Info
---------------------------------------------
Path         type                   <class 'pathlib.Path'>
accumulate   function               <function accumulate at 0x10ab61120>
ipython      ZMQInteractiveShell    <ipykernel.zmqshell.ZMQIn<...>ll object at 0x10a71a9f0>
sys          module                 <module 'sys' (built-in)>
total        int                    333332833333500000


In [10]:
!python --version && echo "--- 셸 명령은 ! 로 ---"

zsh:1: command not found: python


### autoreload — 이 저장소에서 실제로 쓰는 설정

`nbtools/` 같은 로컬 모듈을 고쳤을 때, 커널을 재시작하지 않아도 반영되게 한다.
```python
%load_ext autoreload
%autoreload 2
```

## 4. 도움말 보기

| 입력 | 결과 |
|---|---|
| `함수?` | 독스트링 |
| `함수??` | 소스 코드까지 |
| `Tab` | 자동완성 |
| `Shift+Tab` | 커서 위치 함수의 시그니처 |

`?` 는 별도 창(pager)으로 뜨기 때문에, 문서에 남기려면 아래처럼 쓰는 편이 낫다.

In [11]:
import inspect

print(inspect.signature(sorted))
print(sorted.__doc__.splitlines()[0])

(iterable, /, *, key=None, reverse=False)
Return a new list containing all items from the iterable in ascending order.


## 5. 노트북 위생 — 버전관리의 문제

`.ipynb` 는 **JSON** 이다. 셀 소스뿐 아니라 출력·실행 카운터·이미지(base64)까지 들어간다.
그래서 코드 한 줄만 고쳐도 diff 가 수천 줄이 되고, 이미지가 들어가면 저장소가 무거워진다.

실제로 이 저장소가 어떤 상태였는지 보자.

In [12]:
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nbtools").is_dir())
legacy = sorted((ROOT / "legacy").glob("*.ipynb"))

print(f"{'파일':<34}{'셀':>4}{'출력바이트':>11}  커널/버전")
for path in legacy:
    nb = json.loads(path.read_text(encoding="utf-8"))
    out_bytes = sum(len(json.dumps(c.get("outputs", []))) for c in nb["cells"])
    meta = nb.get("metadata", {})
    version = meta.get("language_info", {}).get("version", "?")
    print(f"{path.name:<34}{len(nb['cells']):>4}{out_bytes:>11,}  py {version}")

파일                                   셀      출력바이트  커널/버전
 근사화.ipynb                           6         12  py 3.7.3
News screpion.ipynb                 12      5,100  py 3.7.5
Untitled.ipynb                      13     39,219  py 3.7.5
Untitled1.ipynb                     10        556  py 3.7.5
Untitled2.ipynb                      3        398  py 3.7.5
Untitled3.ipynb                      7      2,781  py 3.7.3
Untitled4.ipynb                      9        731  py 3.7.3
Untitled5.ipynb                     10      5,288  py 3.7.5
Untitled6.ipynb                      8         79  py 3.7.5
cv2 tes 2.ipynb                      2        107  py 3.7.5
cv2 test 1.ipynb                    11      3,964  py 3.7.5
cv2 test.ipynb                      18    159,615  py 3.7.5
image 끝 기준으로 자르기.ipynb              14      2,313  py 3.7.5
image_marsking.ipynb                 7      5,438  py 3.7.3
영역 크기.ipynb                          7         14  py 3.7.3
이미지 회전.ipynb                         8     

보이는 것:
* 노트북마다 **파이썬 버전이 제각각**(3.7.3 / 3.7.5) — 재현 환경이 기록되지 않았다는 뜻
* 출력이 통째로 커밋돼 있다 (한 파일은 수십만 바이트가 출력)

### 대응
| 문제 | 대응 |
|---|---|
| 거대한 diff | `nbstripout` 으로 커밋 시 출력 제거 (`pip install nbstripout && nbstripout --install`) |
| 코드 리뷰 불가 | `jupytext` 로 `.py` 와 쌍(pair)을 이뤄 텍스트로 리뷰 |
| `.ipynb_checkpoints` 커밋 | `.gitignore` 에 추가 (이 저장소도 그렇게 고쳤다) |
| `.DS_Store`, `.idea/` 커밋 | 같은 방식으로 제외 |
| `Untitled3.ipynb` | 파일명이 곧 문서다. 처음부터 이름을 붙인다 |

> **이 저장소의 선택**: 학습 자료라 GitHub 미리보기에서 결과가 보이는 편이 좋으므로
> 출력을 **일부러 커밋**한다. 대신 모든 노트북은 `Restart & Run All` 로 위에서부터
> 한 번에 실행된 상태이고, 결과가 매번 같도록 난수 시드를 고정했다.

## 6. 실습 데이터 준비

이 저장소의 실습 데이터는 내려받지 않고 **코드로 만든다**. 개인 PC 경로에 의존하지 않고,
네트워크 없이도 첫 실행이 되게 하기 위해서다.

In [13]:
sys.path.insert(0, str(ROOT))
from nbtools import ensure_data  # noqa: E402

data_dir = ensure_data()
for file in sorted(data_dir.iterdir()):
    print(f"{file.name:<24}{file.stat().st_size:>9,} bytes")

news_sample.html            8,057 bytes
spec_table.html               661 bytes
used_cars_raw.csv          28,993 bytes


## 정리

* 노트북의 결과는 **실행 순서**에 좌우된다 → 공유 전 `Restart & Run All`
* `sys.executable` 로 커널이 어떤 파이썬인지 항상 확인
* 성능 비교는 `%%timeit`, 한 번 재기는 `%%time`
* `.ipynb` 는 JSON — 출력·체크포인트·IDE 설정을 커밋에 흘리지 않는다

다음: **02. NumPy 핵심** — pandas 와 OpenCV 가 모두 올라타 있는 기반.